In [12]:
pip install "midigpt[inference]"

In [ ]:
pip install miditok

In [21]:
pip install pretty_midi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.0 MB/s eta 0:00:00


In [ ]:
from miditok import REMI, TokenizerConfig
from miditok.utils import split_files_for_training
from miditok.pytorch_data import DatasetMIDI, DataCollator
from torch.utils.data import DataLoader
from pathlib import Path

from miditok import REMI
tokenizer = REMI(params="/content/MIDI-LSTM-and-transformer-decoder/trained_tokenizer/tokenizer_final.json")

In [ ]:
tokenizer.vocab_size

In [ ]:
import zipfile
import os

def extract_datasets(zip_path,extract_path):
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      erro = zip_ref.testzip()
      return zip_ref.extractall(extract_path)

In [ ]:
test_unziped = extract_datasets("/content/MIDI-LSTM-and-transformer-decoder/features/dataset_test.zip","/content/dataset_test")

In [ ]:
test_path = list(Path("/content/dataset_test").glob("**/*.mid"))


In [ ]:
import os

quantidade = len([f for f in os.listdir(test_path) if os.path.isfile(f)])

In [ ]:
import random

def generate_sample_input_predict(test_path, quantidade):
  valor=random.randint(1, quantidade)
  counter = 1
  for index, path in enumerate(test_path,1):
    if counter == valor:
      return path
    counter=counter+1

In [ ]:
musica_sorteada = generate_sample_input_predict(test_path, quantidade)

In [23]:
import pretty_midi

def cortar_midi_pronto(arquivo_entrada, arquivo_saida, inicio, fim):
    pm = pretty_midi.PrettyMIDI(arquivo_entrada)

    pm.adjust_times([inicio, fim], [0, fim - inicio])

    pm.write(arquivo_saida)


In [35]:
from midigpt import Score, Track, Bar
from midigpt.inference import InferenceEngine, GenerationRequest, InferenceConfig, TrackPrompt

engine = InferenceEngine.from_pretrained("yellow_medium")

score = Score(tracks=[Track(bars=[Bar() for _ in range(30)])])

result = engine.session(
    score,
    GenerationRequest(
        tracks=[TrackPrompt(id=0, bars=list(range(30)),
                attributes={"max_polyphony": 3},
                controls={"time_signature": 0})],
        config=InferenceConfig(model_dim=8, mask_mode="attention", temperature=0.8,top_p=0.95,max_attempts=10,silence_check=False,
        novelty_check=False),
    ),
).run()

total = sum(len(b.notes) for t in result.tracks for b in t.bars)
print(f"Generated {total} notes")
result.to_midi("/content/musica1-gpt.mid")




100%|██████████| 30/30 [00:55<00:00,  1.84s/it]

Generated 795 notes


In [27]:
musica_sorteada=cortar_midi_pronto("/content/musica1-gpt.mid", 'musica1-gpt-cortada.mid', 0, 5.0)

In [36]:

score = Score.from_midi  ("/content/musica1-gpt.mid")

request = GenerationRequest(
    tracks=[
        TrackPrompt(id=0, bars=[0,7,15,25]),   # bars to regenerate
        #TrackPrompt(id=1, bars=[], ignore=True), # leave track 1 unchanged
    ],
    config=InferenceConfig(model_dim=8, mask_mode="attention", temperature=0.8),
)

result = engine.session(score, request).run()
result.to_midi("/content/musica2-gpt.mid")

100%|██████████| 4/4 [00:09<00:00,  2.28s/it]
